<!-- # Intro To Agents Part 1: Tools, MCPs & Tracing
--------------------

1. Introduction

2. Agents & Tools with LangChain

3. Model Context Protocols (MCPs) with FastMCP

4. Agents & Tracing with LangSmith

5. Next Steps -->

## Agents, Tools, MCPs and All That
---------


[Div, Grad, Curl and All that](https://www.google.com/books/edition/Div_Grad_Curl_and_All_that/sembQgAACAAJ?hl=en)

In [1]:
import sys
import os
import json
from dotenv import load_dotenv, find_dotenv
load_dotenv(find_dotenv())

True

## 0. Introduction

## 1. Agents

In [2]:
from langchain_groq import ChatGroq
from langchain_core.output_parsers import StrOutputParser
from langchain.agents import create_agent

model = ChatGroq(model="openai/gpt-oss-120b")
llm  = model | StrOutputParser() 

llm.invoke("Hello")

/Users/mikeharmon/Desktop/mcpweather/.venv/lib/python3.13/site-packages/langgraph/checkpoint/serde/encrypted.py:5: LangChainPendingDeprecationWarning: The default value of `allowed_objects` will change in a future version. Pass an explicit value (e.g., allowed_objects='messages' or allowed_objects='core') to suppress this warning.
  from langgraph.checkpoint.serde.jsonplus import JsonPlusSerializer


'Hello! How can I assist you today?'

In [3]:
print(llm.invoke("What is the weather like in New York City today?"))

I’m not able to access real‑time data, so I can’t give you the exact weather conditions for New York City right now. For the most accurate and up‑to‑date forecast, I recommend checking a reliable weather service such as:

- **National Weather Service** (weather.gov)  
- **The Weather Channel** (weather.com)  
- **AccuWeather** (accuweather.com)  
- A weather app on your smartphone (e.g., Apple Weather, Google Weather)

These sources will provide current temperature, precipitation chances, wind, humidity, and any alerts that may be in effect. If you let me know what kind of activities you have planned, I can also offer general advice on what to expect for this time of year in NYC.


<!-- ## 2. Agents & Tools with LangChain -->

## 2. Agents & Tools

In [4]:
PROJECT_ROOT = os.path.abspath('..')
sys.path.insert(0, PROJECT_ROOT)
from mymcp.server import get_weather

ModuleNotFoundError: No module named 'utils'

In [ ]:
results = get_weather("New York")

In [5]:
results

NameError: name 'results' is not defined

In [6]:
from langchain.tools import tool


In [7]:
weather_tool = tool("get_weather", get_weather)

NameError: name 'get_weather' is not defined

In [8]:
type(weather_tool)

NameError: name 'weather_tool' is not defined

In [14]:
await weather_tool.ainvoke("New York")

{'coord': {'lon': -74.006, 'lat': 40.7143},
 'weather': [{'id': 500,
   'main': 'Rain',
   'description': 'light rain',
   'icon': '10n'}],
 'base': 'stations',
 'main': {'temp': 18.78,
  'feels_like': 19.17,
  'temp_min': 17.86,
  'temp_max': 19.76,
  'pressure': 1015,
  'humidity': 94,
  'sea_level': 1015,
  'grnd_level': 1014},
 'visibility': 10000,
 'wind': {'speed': 11.62, 'deg': 313, 'gust': 14.31},
 'rain': {'1h': 0.13},
 'clouds': {'all': 100},
 'dt': 1787274147,
 'sys': {'type': 2,
  'id': 2111743,
  'country': 'US',
  'sunrise': 1787220682,
  'sunset': 1787269669},
 'timezone': -14400,
 'id': 5128581,
 'name': 'New York',
 'cod': 200}

In [9]:
agents = create_agent(model=model, tools=[weather_tool])

NameError: name 'weather_tool' is not defined

In [10]:
query = "What is the weather in New York?"

In [17]:
result = await agents.ainvoke({'messages': [{'role': 'user', 'content': query}]})

In [18]:
messages = result.get("messages")

In [19]:
messages

[HumanMessage(content='What is the weather in New York?', additional_kwargs={}, response_metadata={}, id='b6ccf503-8d73-42a6-be95-ab1949558b0f'),
 AIMessage(content='', additional_kwargs={'reasoning_content': "User asks for weather in New York. We can use get_weather function. Let's call it.", 'tool_calls': [{'id': 'fc_6b0e4f46-0051-4a29-b97b-1422042ec7f8', 'function': {'arguments': '{"city":"New York"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 48, 'prompt_tokens': 261, 'total_tokens': 309, 'completion_time': 0.099997548, 'completion_tokens_details': {'reasoning_tokens': 20}, 'prompt_time': 0.010019718, 'prompt_tokens_details': None, 'queue_time': 0.084205908, 'total_time': 0.110017266}, 'model_name': 'openai/gpt-oss-120b', 'system_fingerprint': 'fp_017482bd7f', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None, 'model_provider': 'groq'}, id='lc_run--01a021da-d747-7c81-bc42-3be4e97c8df5-0', tool_c

<!-- ## 3. Model Context Protocols (MCPs) with FastMCP -->

## 3. Model Context Protocols (MCPs)

In [21]:
from fastmcp import Client

In [22]:
mcp_client = Client("http://localhost:8000/mcp")

In [23]:
async with mcp_client:
    tools = await mcp_client.list_tools()

In [24]:
tools

[Tool(name='get_weather', title=None, description='Fetch current weather for *city* from OpenWeatherMap.\n\nThe function reads the API key from the ``OPEN_WEATHER_MAP_API_KEY``\nenvironment variable unless an explicit ``api_key`` argument is supplied.', inputSchema={'additionalProperties': False, 'properties': {'city': {'type': 'string', 'description': 'City name to query, e.g. ``"London"``.'}, 'api_key': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional explicit API key; if omitted the environment variable\nis used.'}}, 'required': ['city'], 'type': 'object'}, outputSchema={'additionalProperties': True, 'type': 'object'}, icons=None, annotations=None, meta={'fastmcp': {'tags': []}}, execution=None),
 Tool(name='convert_address_to_point', title=None, description='Convert a street address to a Shapely Point object.', inputSchema={'additionalProperties': False, 'properties': {'address': {'type': 'string', 'description': 'A free‑form US address (e

In [25]:
async with mcp_client:
    result = await mcp_client.call_tool("get_weather", {"city": "New York"})

In [26]:
async with mcp_client:
    point = await mcp_client.call_tool("convert_address_to_point", {"address": "567 Ocean Avenue, Brooklyn, NY 11226"})

In [27]:
from mymcp.server import find_closest_restroom

ModuleNotFoundError: No module named 'utils'

In [31]:
async with mcp_client:
    result = await mcp_client.call_tool("find_closest_restroom", {"point": json.loads(point.content[0].text)})

In [28]:
find_closest_restroom(json.loads(point.content[0].text))

NameError: name 'find_closest_restroom' is not defined

In [33]:
async with mcp_client:
    result = await mcp_client.call_tool("get_police_precinct", {"point": json.loads(point.content[0].text)})

In [34]:
result

CallToolResult(content=[TextContent(type='text', text='70', annotations=None, meta=None)], structured_content={'result': 70}, meta={'fastmcp': {'wrap_result': True}}, data=70, is_error=False)

## 4. Agents & MCPs

In [35]:
from langchain_mcp_adapters.client import MultiServerMCPClient  

lc_client = MultiServerMCPClient({"config": {"url": "http://localhost:8000/mcp", "transport": "http"}})

In [37]:
tool_list = await lc_client.get_tools()

In [30]:
tool_list

[StructuredTool(name='get_weather', description='Fetch current weather for *city* from OpenWeatherMap.\n\nThe function reads the API key from the ``OPEN_WEATHER_MAP_API_KEY``\nenvironment variable unless an explicit ``api_key`` argument is supplied.', args_schema={'additionalProperties': False, 'properties': {'city': {'type': 'string', 'description': 'City name to query, e.g. ``"London"``.'}, 'api_key': {'anyOf': [{'type': 'string'}, {'type': 'null'}], 'default': None, 'description': 'Optional explicit API key; if omitted the environment variable\nis used.'}}, 'required': ['city'], 'type': 'object'}, metadata={'_meta': {'fastmcp': {'tags': []}}}, response_format='content_and_artifact', coroutine=<function convert_mcp_tool_to_langchain_tool.<locals>.call_tool at 0x10eb3cc20>),
 StructuredTool(name='convert_address_to_point', description='Convert a street address to a Shapely Point object.', args_schema={'additionalProperties': False, 'properties': {'address': {'type': 'string', 'descrip

In [38]:
agent = create_agent(model=model, tools=tool_list)

In [39]:
result = await agent.ainvoke({
    'messages': [{'role': 'user', 
                 'content': 'What police precinct is 306 West 54th Street, manhattan, manhattan in?'
}]})

In [42]:
print(result.get("messages")[-1].content)

The address 306 West 54th Street, Manhattan, NY is in **NYPD Precinct 18 – Midtown North Precinct**.  

- **Borough:** Manhattan  
- **Phone:** 212‑767‑8400  
- **Address:** 306 West 54th Street, New York, NY 10019.


In [44]:
result = await agent.ainvoke({
    'messages': [{'role': 'user', 
                 'content': 'Where is the closet bathroom to 625 Atlantic Ave, Brooklyn?'
}]})
print(result.get("messages")[-1].content)

ToolException: Error calling tool 'find_closest_restroom': 'float' object has no attribute 'get'

In [45]:
result = await agent.ainvoke({
    'messages': [{'role': 'user', 
                 'content': 'Whats the temparture in Boston?'
}]})
print(result.get("messages")[-1].content)

The current weather in **Boston, MA** (as reported by OpenWeatherMap) is:

- **Temperature:** **21.7 °C** (≈ 71 °F)  
- **Feels like:** 22.2 °C (≈ 72 °F)  
- **Conditions:** Moderate rain, with clouds covering the sky (100 % cloud cover)  
- **Wind:** 3.6 m/s from the northwest (≈ 12 km/h)  
- **Humidity:** 84 %  
- **Rainfall (last hour):** 2.3 mm  

So it’s a cool, rainy day in Boston right now. Stay dry!


<!--  -->

## 5. Agent Traces in Langsmith

## 6. Next Steps